In [ ]:
# §36i smoke test: reconstruct SpikingEncoderV2 (3-layer hxsnn.Synapse+LIF)
# and compare forward-pass output to the local spiking_jepa_v2.py encoder
# on identical input. mock=True, hxtorch.init_hardware() never called
# (zero hardware quota consumed). See RESEARCH_NOTES.md §36i / §49.
import torch, numpy as np
import hxtorch
import hxtorch.snn as hxsnn

In [ ]:
# Ground truth: weights + input + expected output, generated locally
# from spiking_jepa_v2.py's SpikingEncoderV2 (see /tmp/gen_groundtruth_small.py).
# Binary (unsigned) spike input, NOT signed sigma-delta (blocker §36i#1 fix).
GT = {'in_dim': 8, 'hidden_dim': 6, 'out_dim': 4, 'T': 10, 'batch': 1, 'x': [[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [1.0, 0.0, 0.0, 1.0, 0.0, 1.0, 0.0, 1.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 1.0, 1.0, 0.0, 1.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 1.0, 0.0, 1.0, 0.0, 0.0], [0.0, 0.0, 0.0, 1.0, 1.0, 1.0, 0.0, 0.0], [0.0, 1.0, 1.0, 1.0, 1.0, 0.0, 0.0, 0.0]], 'fc1_weight': [[-0.182584, -0.241022, 0.187588, -0.142908, 0.21458, -0.083898, 0.202249, -0.2747], [-0.178421, 0.10779, 0.074744, -0.090142, 0.210742, 0.240349, -0.256388, -0.188751], [0.323735, -0.1193, -0.12534, -0.342096, -0.202469, 0.088319, -0.046667, -0.256639], [0.008293, -0.241506, -0.299954, -0.194689, -0.309434, -0.225121, 0.353415, 0.066777], [0.108951, -0.329754, -0.232205, -0.117682, 0.055286, -0.311099, -0.152337, -0.211661], [0.00098, -0.131558, -0.0245, -0.239578, -0.242677, -0.206264, -0.12102, -0.279053]], 'fc1_bias': [0.296444, -0.070168, 0.304196, 0.110161, -0.299388, 0.244671], 'fc2_weight': [[-0.112327, -0.156492, -0.338875, -0.405864, 0.116804, -0.089179], [0.158941, -0.335034, 0.303095, -0.299676, -0.070494, 0.085271], [0.210759, 0.329583, 0.371898, -0.323709, 0.102743, -0.175598], [-0.044738, -0.30557, 0.371856, -0.299634, 0.218189, 0.143475], [0.132663, -0.220718, 0.371078, 0.089713, 0.052517, -0.359771], [0.171378, -0.061246, -0.187028, 0.350663, 0.091018, -0.225874]], 'fc2_bias': [-0.20663, -0.019496, 0.22795, -0.104321, -0.232936, -0.139802], 'fc3_weight': [[-0.304992, 0.145595, 0.316001, -0.384341, 0.094816, 0.210898], [0.074027, -0.145387, 0.213082, 0.21454, 0.152655, -0.071738], [-0.108105, 0.043675, -0.072119, -0.121658, 0.260955, 0.350848], [-0.040416, -0.091406, 0.005957, -0.024376, 0.098147, 0.114405]], 'fc3_bias': [-0.370794, -0.150659, 0.343798, 0.159035], 'expected_repr_out': [0.0, 0.0, 0.7, 0.3], 'expected_spike_rate': 0.1625, 'expected_layer_spike_rates': [0.1, 0.166667, 0.25]}

IN_DIM, HID, OUT = GT['in_dim'], GT['hidden_dim'], GT['out_dim']
T, BATCH = GT['T'], GT['batch']
WEIGHT_SCALE = 55.  # reused from ts_12 tutorial's own documented example value

In [ ]:
exp = hxsnn.Experiment(dt=1e-6, mock=True)

syn1 = hxsnn.Synapse(IN_DIM, HID, exp)
syn1.weight.data = torch.tensor(GT['fc1_weight']) * WEIGHT_SCALE
lif1 = hxsnn.LIF(HID, exp, tau_mem=4.48e-6, tau_syn=4.48e-6,
                  enable_cadc_recording=True, enable_spike_recording=True)

syn2 = hxsnn.Synapse(HID, HID, exp)
syn2.weight.data = torch.tensor(GT['fc2_weight']) * WEIGHT_SCALE
lif2 = hxsnn.LIF(HID, exp, tau_mem=4.48e-6, tau_syn=4.48e-6,
                  enable_cadc_recording=True, enable_spike_recording=True)

syn3 = hxsnn.Synapse(HID, OUT, exp)
syn3.weight.data = torch.tensor(GT['fc3_weight']) * WEIGHT_SCALE
lif3 = hxsnn.LIF(OUT, exp, tau_mem=4.48e-6, tau_syn=4.48e-6,
                  enable_cadc_recording=True, enable_spike_recording=True)

In [ ]:
# x: (T, IN_DIM) for batch=1 -> (T, batch, IN_DIM)
x = torch.tensor(GT['x']).unsqueeze(1)
print("input shape:", x.shape)

# IMPORTANT (lesson from first attempt, see RESEARCH_NOTES.md §49/§50):
# pass the Handle object itself between layers (z1, not z1.spikes) -
# .spikes is an unresolved promise until hxsnn.run() executes, so reading
# it early bakes in None permanently.
g1 = syn1(hxsnn.LIFObservables(spikes=x))
z1 = lif1(g1)
g2 = syn2(z1)
z2 = lif2(g2)
g3 = syn3(z2)
z3 = lif3(g3)

hxsnn.run(exp, T)

In [ ]:
spk1, spk2, spk3 = z1.spikes, z2.spikes, z3.spikes
print("RESULT spk3.spikes shape:", tuple(spk3.shape))

repr_out = spk3.sum(0)[0] / T  # sum over time, batch 0, /T -> matches local repr_out def
print("RESULT repr_out:", repr_out.tolist())

n_spikes = spk1.sum() + spk2.sum() + spk3.sum()
n_possible = spk1.numel() + spk2.numel() + spk3.numel()
print("RESULT spike_rate:", (n_spikes / n_possible).item())
print("RESULT layer_spike_rates:", [
    (spk1.sum() / spk1.numel()).item(),
    (spk2.sum() / spk2.numel()).item(),
    (spk3.sum() / spk3.numel()).item(),
])

print()
print("EXPECTED repr_out:", GT['expected_repr_out'])
print("EXPECTED spike_rate:", GT['expected_spike_rate'])
print("EXPECTED layer_spike_rates:", GT['expected_layer_spike_rates'])